<a href="https://colab.research.google.com/github/RatchanonPa/Data-Warehouse-and-Big-Data-Analytics/blob/main/Hackathon_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# --- Mount Google Drive (Optional, if your data is on Drive) ---
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# prompt: un zip /content/drive/MyDrive/io-t-sleep-stage-classification-version-2.zip

!unzip /content/drive/MyDrive/io-t-sleep-stage-classification-version-2.zip

In [2]:
from google.colab import drive
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from sklearn.preprocessing import StandardScaler, LabelEncoder

In [4]:
import pandas as pd
import glob
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.preprocessing import LabelEncoder, StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, LSTM, Dense, Dropout, BatchNormalization, MaxPooling1D, Input
from tensorflow.keras.callbacks import EarlyStopping, Callback
from sklearn.metrics import f1_score
from sklearn.utils import class_weight  # For class weights

In [ ]:
drive.mount('/content/drive')
!unzip /content/drive/MyDrive/io-t-sleep-stage-classification-version-2.zip

In [5]:
# โหลดข้อมูลจากไฟล์ CSV
train_path = "/content/train/train"
train_csv_files = sorted(glob.glob(f"{train_path}/*.csv"))

# อ่านข้อมูลและจัดรูปแบบใหม่
train_data = []
train_labels = []
for file in train_csv_files:
    df = pd.read_csv(file)
    # แยก features และ labels
    features = df.drop('Sleep_Stage', axis=1).values
    labels = df['Sleep_Stage'].values[::480]  # 1 label ต่อ 480 timestep

    train_data.append(features)
    train_labels.extend(labels)

# รวมข้อมูลและปรับขนาด
X_train = np.concatenate(train_data)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_train = X_train.reshape(-1, 480, X_train.shape[1])  # (samples, timesteps, features)

# เข้ารหัส labels
le = LabelEncoder()
y_train = le.fit_transform(train_labels)
y_train = tf.keras.utils.to_categorical(y_train)

In [6]:
model = models.Sequential([
    layers.Input(shape=(480, 8)),
    layers.Conv1D(32, 3, activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling1D(2),
    layers.LSTM(64, return_sequences=True),
    layers.LSTM(64),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(len(le.classes_), activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# ฝึกโมเดล
history = model.fit(
    X_train, y_train,
    epochs=100,
    batch_size=64,
    validation_split=0.2,
    callbacks=[callbacks.EarlyStopping(patience=10)]
)

Epoch 1/100
831/831 ━━━━━━━━━━━━━━━━━━━━ 28s 27ms/step - accuracy: 0.6670 - loss: 0.8114 - val_accuracy: 0.6281 - val_loss: 0.8762
Epoch 2/100
831/831 ━━━━━━━━━━━━━━━━━━━━ 38s 26ms/step - accuracy: 0.7008 - loss: 0.7265 - val_accuracy: 0.6131 - val_loss: 0.9024
Epoch 3/100
831/831 ━━━━━━━━━━━━━━━━━━━━ 40s 25ms/step - accuracy: 0.7163 - loss: 0.6813 - val_accuracy: 0.6157 - val_loss: 0.9379
Epoch 4/100
831/831 ━━━━━━━━━━━━━━━━━━━━ 40s 24ms/step - accuracy: 0.7200 - loss: 0.6640 - val_accuracy: 0.5970 - val_loss: 0.9715
Epoch 5/100
831/831 ━━━━━━━━━━━━━━━━━━━━ 22s 26ms/step - accuracy: 0.7327 - loss: 0.6426 - val_accuracy: 0.6103 - val_loss: 0.9837
Epoch 6/100
831/831 ━━━━━━━━━━━━━━━━━━━━ 20s 24ms/step - accuracy: 0.7344 - loss: 0.6409 - val_accuracy: 0.5883 - val_loss: 1.0259
Epoch 7/100
831/831 ━━━━━━━━━━━━━━━━━━━━ 20s 24ms/step - accuracy: 0.7383 - loss: 0.6289 - val_accuracy: 0.5827 - val_loss: 0.9905
Epoch 8/100
831/831 ━━━━━━━━━━━━━━━━━━━━ 20s 23ms/step - accuracy: 0.7479 - loss: 0

In [7]:
# โหลดข้อมูลทดสอบ
test_dirs = sorted(glob.glob("/content/test_segment/test_segment/test*"))

all_segments = []
all_ids = []

for dir_path in test_dirs:
    # อ่านข้อมูลและปรับขนาด
    dir_data = []
    for file in sorted(glob.glob(f"{dir_path}/*.csv")):
        df = pd.read_csv(file)
        dir_data.append(df.values)

    dir_data = np.concatenate(dir_data)
    dir_data = scaler.transform(dir_data)  # ใช้ scaler จากชุดฝึก

    # เติมข้อมูลให้ครบ 480 timestep
    if len(dir_data) % 480 != 0:
        padding = np.zeros((480 - len(dir_data)%480, 8))
        dir_data = np.vstack([dir_data, padding])

    # สร้าง ID
    num_segments = len(dir_data) // 480
    dir_ids = [f"{dir_path.split('/')[-1]}_{i:05d}" for i in range(num_segments)]

    all_segments.append(dir_data.reshape(-1, 480, 8))
    all_ids.extend(dir_ids)

# รวมข้อมูลทดสอบทั้งหมด
X_test = np.concatenate(all_segments)

In [11]:
# ทำนายผล
predictions = model.predict(X_test)
predicted_classes = np.argmax(predictions, axis=1)
predicted_labels = le.inverse_transform(predicted_classes)

# สร้าง DataFrame และบันทึกไฟล์
submission_df = pd.DataFrame({
    'id': all_ids,
    'labels': predicted_labels
})
submission_df.to_csv('submission1.csv', index=False)

220/220 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step


In [12]:
print("Submission file created successfully!")
print("Unique labels in submission:", submission_df['labels'].unique())

Submission file created successfully!
Unique labels in submission: ['N' 'W' 'R']
